# NB_patient_360

Builds `adb_mediflow.gold.patient_360` (one row per patient) from silver patients, hospitals, insurance providers and aggregated lab results, per the Silver to Gold STTM. Full rebuild every run.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_mediflow.gold.patient_360 AS
WITH lab AS (
  SELECT
    patient_id,
    count(*)                             AS total_tests,
    count_if(result_status = 'Abnormal') AS abnormal_test_count,
    count_if(result_status = 'Critical') AS critical_test_count,
    max(test_date)                       AS last_test_date
  FROM adb_mediflow.silver.lab_results
  GROUP BY patient_id
)
SELECT
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                              AS date_of_birth,
  CAST(floor(datediff(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  p.registered_hospital_id                    AS hospital_id,
  h.hospital_name,
  p.insurance_provider_id,
  i.provider_name                             AS insurance_provider_name,
  CAST(coalesce(l.total_tests, 0) AS INT)         AS total_tests,
  CAST(coalesce(l.abnormal_test_count, 0) AS INT) AS abnormal_test_count,
  CAST(coalesce(l.critical_test_count, 0) AS INT) AS critical_test_count,
  l.last_test_date
FROM adb_mediflow.silver.patients p
LEFT JOIN adb_mediflow.silver.hospitals h           ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_mediflow.silver.insurance_providers i ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab l                                     ON p.patient_id = l.patient_id;

In [0]:
# %sql
# SELECT count(*) AS row_count, count(DISTINCT patient_id) AS distinct_patients,
#        sum(total_tests) AS total_tests, count_if(hospital_name IS NULL) AS missing_hospital
# FROM adb_mediflow.gold.patient_360;

In [0]:
updated_df = spark.table("adb_mediflow.gold.patient_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))